# XGBoost Next-Day Return on SPY

Gradient-boosted regressor predicting next-day SPY log return from technical features. Chronological 80/20 split; results reported on the held-out OOS window.

See [`README.md`](README.md) for the full writeup and [`train.py`](train.py) / [`backtest.py`](backtest.py) for the reproducible CLI versions.

In [ ]:
import joblib
import matplotlib.pyplot as plt
import pandas as pd
from train import END, RESULTS, START, SYMBOL, build_features

from tradinglib.backtest import run_backtest
from tradinglib.loaders.equities.yfinance import load_daily

## Load data and features

In [ ]:
bars = load_daily(SYMBOL, start=START, end=END)
prices = bars["close"]
feats = build_features(prices)
feats.tail()

## Load the trained model

Run `python train.py` first if `results/model.joblib` doesn't exist yet.

In [ ]:
bundle = joblib.load(RESULTS / "model.joblib")
model = bundle["model"]
features = bundle["features"]
split = bundle["split_index"]
print(f"Train/test split at {split.date()} — features: {features}")

## Predict on the OOS window and form a signal

In [ ]:
x_oos = feats.loc[feats.index >= split, features].dropna()
pred = pd.Series(model.predict(x_oos), index=x_oos.index)
signal = (pred > 0).astype(float)
signal.value_counts().rename(index={0.0: "flat", 1.0: "long"})

## Backtest on the OOS window

In [ ]:
oos_prices = prices.loc[signal.index]
result = run_backtest(oos_prices, signal, fee_bps=1.0, slippage_bps=0.5)
pd.Series(result.metrics)

## Equity curve vs buy & hold

In [ ]:
buy_hold = (1.0 + oos_prices.pct_change().fillna(0.0)).cumprod() * result.config["initial_capital"]

fig, ax = plt.subplots(figsize=(12, 5))
result.equity_curve.plot(ax=ax, label="XGBoost (OOS)")
buy_hold.plot(ax=ax, label="Buy & hold (OOS)", alpha=0.6)
ax.set_title(f"{SYMBOL} \u2014 XGBoost next-day return (OOS)")
ax.set_ylabel("Equity ($)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## Feature importance

In [ ]:
imp = pd.Series(model.feature_importances_, index=features).sort_values()
imp.plot.barh(figsize=(8, 4))
plt.title("XGBoost feature importance")
plt.tight_layout()
plt.show()